# 第5章 事故候选地点筛查

本工作本是一份连续的实践，不另分学生任务版与参考版。按单元逐步运行，在参数区修改条件，记录自己的结果和解释；不要只执行整章脚本后截一张图。

**协议：** `ch05-spatial-v1`  
**必做：** 固定500米网格；高斯KDE；DBSCAN  
**对象：** 局部距离：米；KDE概率密度：1/km²  
**样本：** 2024年1月7542事故，7068有效点；固定2069个有记录500米网格中心  
**划分：** 空间探索，无独立热点真值；不报告识别准确率  
**比较：** 同一有效点集和固定评价位置；h=250/500/1000米；eps=150/350/700米，min_samples=5/15

先解压完整资料包，在其目录内运行。安装 `python -m pip install -r chapters/python/learning-requirements.txt`，再启动Jupyter。代码只读取固定真实数据；输出写入`outputs/chXX/`，不覆盖原数据。

每一步的“请解释”需用本次实际结果回答。默认参考配置可直接运行，但自动生成文件不代表学生已完成分析。


In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'chapters/data').is_dir() and (p/'projects/data').is_dir())
sys.path.insert(0, str(ROOT/'chapters/python'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from threadpoolctl import threadpool_limits
from learning_support import metrics, metric_table, csv_file, primary, predictions_table, report, save_plot
THREADS = threadpool_limits(limits=1)
print('Data root:', ROOT)


## 1. 固定有效点集与距离单位
不要对真实未知坐标进行插值补造。此局部距离近似服务教学对照，正式工程需要适当投影与误差检查。


In [ ]:
FILES=['projects/data/crashes.json']
raw=json.loads((ROOT/FILES[0]).read_text(encoding='utf-8'))['rows']
valid=[r for r in raw if r[7]]
xy=np.array([[(r[4]+74.3)*111320*np.cos(np.deg2rad(40.73)),(r[3]-40.45)*111320] for r in valid])
assert len(raw)==7542 and len(valid)==7068
cells=sorted({tuple(map(int,r)) for r in np.floor(xy/500)})
grid_ids=[f'{x}:{y}' for x,y in cells]
probe=(np.array(cells)+.5)*500
print('Total / valid / excluded:',len(raw),len(valid),len(raw)-len(valid),'Probe cells:',len(probe))


## 2. 自己运行KDE带宽比较
KDE输出归一化概率密度，不是具有暴露分母的事故风险。改变BANDWIDTHS后重新执行，而不是只切换现成图片。


In [ ]:
from sklearn.neighbors import KernelDensity
BANDWIDTHS=[250,500,1000]
density={str(h):np.exp(KernelDensity(kernel='gaussian',bandwidth=h).fit(xy).score_samples(probe))*1e6 for h in BANDWIDTHS}
top=lambda v:np.argsort(-v,kind='stable')[:20]
kde_table=pd.DataFrame([[h,float(d.max()),len(set(top(d))&set(top(density['500'])))] for h,d in density.items()],columns=['bandwidth_m','max_density_per_km2','top20_overlap_500m'])
display(kde_table)


## 3. 实际运行DBSCAN并检查巨簇
修改EPS或MIN_SAMPLES，查看哪些配置过碎、哪些产生巨簇。簇少或噪声少不必然更好。


In [ ]:
from sklearn.cluster import DBSCAN
EPS=[150,350,700]
MIN_SAMPLES=[5,15]
clusters={};rows=[]
for eps in EPS:
    for minimum in MIN_SAMPLES:
        labels=DBSCAN(eps=eps,min_samples=minimum).fit_predict(xy)
        clusters[f'{eps}-{minimum}']=labels.tolist()
        sizes=np.bincount(labels[labels>=0])
        rows.append([eps,minimum,len(sizes),int((labels<0).sum()),int(sizes.max(initial=0))])
dbscan_table=pd.DataFrame(rows,columns=['eps_m','min_samples','clusters','noise','largest_cluster'])
display(dbscan_table)


## 4. 候选地图与现场排查清单
先按500米KDE给出3个可回查的候选示例。请编辑SELECTED和SITE_NOTES，写出自己的选择理由；代码不能替你确认道路结构或设施问题。


In [ ]:
from collections import Counter
counts=Counter(tuple(map(int,r)) for r in np.floor(xy/500))
SELECTED=top(density['500'])[:3].tolist()  # 可改成固定评价点索引
SITE_NOTES={}  # 例如 {索引:'结合具体数据写选择依据；道路对象需另核实'}
candidates=[]
for i in SELECTED:
    col,row=cells[i]
    candidates.append({'grid_id':grid_ids[i],'east_m':float(probe[i,0]),'north_m':float(probe[i,1]),'accidents':counts[(col,row)],'density':float(density['500'][i]),'reason':SITE_NOTES.get(i,'自动高密度候选示例，待学生结合事故和道路资料核查'),'road_object':'待匹配路段或交叉口','evidence_needed':'道路结构、设施、交通暴露与现场记录'})
plt.figure(figsize=(8,6));plt.scatter(probe[:,0]/1000,probe[:,1]/1000,c=density['500'],s=5,cmap='Blues');plt.colorbar(label='Probability density / km2')
plt.scatter(probe[SELECTED,0]/1000,probe[SELECTED,1]/1000,c='red',marker='x')
plt.axis('equal');plt.xlabel('Local east / km');plt.ylabel('Local north / km');save_plot(ROOT,5,'candidate_map')
display(pd.DataFrame(candidates))


## 5. 导出空间证据
保存密度、簇成员与候选清单，而不只保存截图。主实验网格ID为“列:行”，以当前固定原点与500米评价网格解释，旧CSV自测另有旧格式说明。


In [ ]:
outputs={'grid_ids':grid_ids,'density':{h:v.tolist() for h,v in density.items()},'clusters':clusters,'candidates':candidates}
primary(ROOT,5,FILES,{'bandwidths':BANDWIDTHS,'eps':EPS,'min_samples':MIN_SAMPLES,'grid_m':500},outputs)
csv_file(ROOT/'outputs/ch05/candidates.csv',candidates[0].keys(),[r.values() for r in candidates])
csv_file(ROOT/'outputs/ch05/dbscan_memberships.csv',['collision_id',*clusters],[[r[0],*[labels[i] for labels in clusters.values()]] for i,r in enumerate(valid)])
report(ROOT,5,'事故候选地点现场核查报告',{'KDE':kde_table.to_string(index=False),'DBSCAN':dbscan_table.to_string(index=False),'候选清单':pd.DataFrame(candidates).to_string(index=False)},['为什么选择这3处候选范围？','参数变化导致哪些结论不稳定？','实际道路排查还缺什么资料？'])
